In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [7]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [7]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [8]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [9]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: Sliding ma więcej wierszy, bo pojedyncze okna nakładają się na siebie, co sprawia, że ta sama transakcja jest liczona wielokrotnie w różnych przedziałach czasowych.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [16]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") grupuje dane dla każdego sklepu osobno, a groupBy(window(...), "store") po sklepach w danym oknie czasowym

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: Są to 3 okna:
   # 1. Okno 08:30 – 09:30
   # 2. Okno 09:00 – 10:00
   # 3. Okno 09:30 – 10:30

In [ ]:
# PRACA DOMOWA

In [11]:
# 1.Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

In [20]:
from pyspark.sql.functions import col, date_format, avg

gdansk_hours = (
    df.filter(col("store") == "Gdańsk") 
    .withColumn("hour", date_format(col("timestamp"), "HH")) 
)

result = (
    gdansk_hours.groupBy("hour") 
    .agg(avg("amount").alias("avg_transaction")) 
    .orderBy("avg_transaction")
)
result.show(1)

+----+-----------------+
|hour|  avg_transaction|
+----+-----------------+
|  08|395.0118407310706|
+----+-----------------+
only showing top 1 row



In [ ]:
# 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.

In [15]:
from pyspark.sql.functions import window, count, col

ilosc_w_oknie = (
    df.groupBy(
        window("timestamp", "30 minutes"), 
        "category"
    )
    .agg(count("tx_id").alias("liczba_tx"))
    .where(
        (col("window.start") == "2026-04-12 09:00:00") & 
        (col("window.end") == "2026-04-12 09:30:00")
    )
)

ilosc_w_oknie.select(
    col("window.start").alias("od"), 
    col("window.end").alias("do"), 
    "category", 
    "liczba_tx"
).show(truncate=False)



+-------------------+-------------------+-----------+---------+
|od                 |do                 |category   |liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|611      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|odzież     |605      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|książki    |622      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|żywność    |567      |
+-------------------+-------------------+-----------+---------+



In [25]:
# 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [17]:
from pyspark.sql.functions import window, count, col, desc

okno_15_min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
)

okno_15_min.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("liczba_tx").alias("max_transakcje")
).show(1, truncate=False)

+-------------------+-------------------+--------------+
|od                 |do                 |max_transakcje|
+-------------------+-------------------+--------------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234          |
+-------------------+-------------------+--------------+
only showing top 1 row



In [1]:
# WCZEŚNIEJSZE ZADANIA Z LABORATORIUM 2

In [7]:
# Zadanie 2.2 - Statystyki per kategoria
# Policz sumę, minimum i maksimum kwoty dla każdej kategorii.

from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

# TWÓJ KOD

statystyki = (
    df.groupBy("category")
    .agg(
        _round(_max("amount"), 2).alias("maksymalna_kwota"),
        _round(_min("amount"), 2).alias("minimalna_kwota"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("category")
)

statystyki.show()

+-----------+----------------+---------------+----------+
|   category|maksymalna_kwota|minimalna_kwota|  suma_PLN|
+-----------+----------------+---------------+----------+
|elektronika|          9999.0|            9.0|1520770.69|
|    książki|         9107.25|            5.0| 851382.08|
|     odzież|         9696.63|            5.0| 849877.55|
|    żywność|         6916.92|            5.0| 789514.43|
+-----------+----------------+---------------+----------+



In [8]:
# Zadanie 3.2 Okna 30-minutowe per sklep
# Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. Posortuj po oknie, a w ramach okna po sklepie.

# TWÓJ KOD
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

wynik = (
    df.groupBy(
        window("timestamp", "30 minutes"), 
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_pln")
    )
    .orderBy("window.start", "store")
)

wynik.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    "store",
    "liczba_tx",
    "suma_pln"
).show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_pln |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [13]:
# Zadanie 3.3 — W której godzinie sklep “Kraków” miał najwyższy przychód?
# Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejąco po sumie.

from pyspark.sql.functions import desc, date_format, avg

# TWÓJ KOD

krakow_hours = (
    df.filter(col("store") == "Kraków")
    .withColumn("hour", date_format(col("timestamp"), "HH")) 
)
result = (
    krakow_hours.groupBy("hour") 
    .agg(avg("amount").alias("avg_transaction")) 
    .orderBy(desc("avg_transaction"))
)
result.show(1)

+----+-----------------+
|hour|  avg_transaction|
+----+-----------------+
|  08|415.7464433617539|
+----+-----------------+
only showing top 1 row

